# Acoustic-peak recovery in the Euclid–DESI overlap
**Biswajit Jana**

Apply the same sky cut to galaxies and matching random points in eight public DESI AbacusSummit mock realisations. Compare the full window, the Euclid overlap, and the overlap thinned to the observed DESI count.

This run streams about 11.5 GB. The FITS streams are reduced in memory and the cache stores less than 120 MB of arrays. The fit is a pre-reconstruction acoustic-peak diagnostic.


In [ ]:
%pip -q install numpy scipy astropy astropy-healpix mocpy matplotlib


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

branch = "research/euclid-dr1-foundation-day01-20261114"
repo = Path("daily-astro-notebooks").resolve()
relative = Path("euclid-dr1-foundation/desi-bao-window")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--branch", branch, "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/Biswajit1999/daily-astro-notebooks.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set", str(relative)], check=True)
study = repo / relative
cache = Path("abacus-lrg-cache").resolve()
output = Path("abacus-recovery").resolve()
output.mkdir(exist_ok=True)
env = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "MPLBACKEND": "Agg"}


## Apply the Euclid sky mask while reading the DESI files


In [ ]:
subprocess.run(
    [sys.executable, str(study / "fetch_mocks.py"), "--cache", str(cache), "--jobs", "4"],
    env=env, check=True,
)


## Count weighted pairs and fit the acoustic contrast


In [ ]:
results = output / "mock_recovery.json"
subprocess.run(
    [sys.executable, str(study / "mock_recovery.py"), "--cache", str(cache),
     "--output", str(results), "--workers", "2"],
    env=env, check=True,
)


In [ ]:
import json
from statistics import median

data = json.loads(results.read_text())
summary = data["summary"]
for i, window in enumerate(data["windows"]):
    counts = [sample["windows"][window]["data_count"] for sample in data["results"]]
    sd = summary["primary_amplitude_sample_sd"][i]
    mean = summary["fixed_template_means"][i][0]
    print(f"{window:23s}  Ngal {median(counts):9.0f}  amplitude {mean:+.5f}  scatter {sd:.5f}")


In [ ]:
from IPython.display import Image, display

figure = output / "mock_recovery.png"
subprocess.run([sys.executable, str(study / "plot_mock_recovery.py"), "--input", str(results),
                "--output", str(figure)], env=env, check=True)
display(Image(filename=str(figure)))


[Methods, limitations and references](https://github.com/Biswajit1999/daily-astro-notebooks/blob/research/euclid-dr1-foundation-day01-20261114/euclid-dr1-foundation/desi-bao-window/MOCK_RECOVERY.md)
